# 23 — Data Provenance

Provenance is AbaQuant's metadata layer that explains **where data came
from**, when it was retrieved or constructed, how cache was used, and what
transformations were applied. This notebook shows `.provenance` attached to
derivative diagnostics, a manual rate curve, a portfolio allocator, a
backtest, a credit assessment, an integrated risk dashboard, and an
exportable report.

**Sections:**
1. Build one object from each domain
2. Inspect `.provenance` on each
3. Merge provenance from multiple sources


## Setup


In [ ]:
import os, sys
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")

def _ensure_abaquant_importable():
    try:
        import abaquant  # noqa: F401
        return
    except ModuleNotFoundError:
        pass
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        src = candidate / "src"
        if (src / "abaquant" / "__init__.py").exists():
            sys.path.insert(0, str(src))
            return
    raise ModuleNotFoundError(
        "Could not import abaquant. Run this notebook from within the AbaQuant "
        "repository (or pip install abaquant)."
    )

_ensure_abaquant_importable()
import abaquant
print(f"AbaQuant version: {abaquant.__version__}")

In [ ]:
import numpy as np
import pandas as pd

from abaquant import RiskDashboard
from abaquant.credit import (
    BalanceSheetInputs,
    CashFlowInputs,
    CreditAnalysisInputs,
    IncomeStatementInputs,
    calculate_credit_proxy_metrics,
)
from abaquant.derivatives.models import BlackScholesMertonModel
from abaquant.portfolio import PortfolioAllocator
from abaquant.rates import RateCurve

## 1. Build one object from each domain


In [ ]:
def deterministic_returns() -> pd.DataFrame:
    index = pd.date_range("2026-01-01", periods=24, freq="D")
    step = np.arange(len(index), dtype=float)
    return pd.DataFrame(
        {
            "NVDA": 0.0010 + np.sin(step / 3.0) * 0.0015,
            "MSFT": 0.0007 + np.cos(step / 4.0) * 0.0010,
            "AAPL": 0.0008 + np.sin(step / 5.0) * 0.0009,
        },
        index=index,
    )

def deterministic_credit_assessment():
    inputs = CreditAnalysisInputs(
        balance_sheet=BalanceSheetInputs(
            total_debt=120.0, total_equity=260.0, current_assets=190.0, inventory=30.0,
            current_liabilities=85.0, cash_and_cash_equivalents=45.0, total_assets=640.0,
            total_liabilities=300.0, retained_earnings=105.0, long_term_debt=100.0,
        ),
        income_statement=IncomeStatementInputs(
            revenue=520.0, gross_profit=270.0, ebit=85.0, ebitda=100.0,
            interest_expense=10.0, net_income=60.0,
        ),
        cash_flow_statement=CashFlowInputs(operating_cash_flow=82.0),
        reporting_currency="USD",
        reporting_period="FY2026",
    )
    return calculate_credit_proxy_metrics(inputs)

option = BlackScholesMertonModel(100.0, 105.0, 1.0, 0.04, 0.22)
diagnostics = option.diagnostics("call")
curve = RateCurve.from_rates({0.5: 0.04, 1.0: 0.045, 2.0: 0.05})

allocator = PortfolioAllocator(deterministic_returns(), annual_risk_free_rate=curve.zero_rate(1.0))
backtest = allocator.backtest(weights="equal_weight", rebalance="weekly", benchmark="equal_weight")
credit = deterministic_credit_assessment()
dashboard = RiskDashboard(
    allocator,
    credit_assessments={"NVDA": credit},
    weights={"NVDA": 1 / 3, "MSFT": 1 / 3, "AAPL": 1 / 3},
    backtest=backtest,
)
report = dashboard.report()

## 2. Inspect `.provenance` on each

Every object above carries `.provenance` — an immutable `DataProvenance`
record with provider, dataset, retrieval timestamp, cache status, and an
ordered list of transformation steps.


In [ ]:
def summarize_provenance(label: str, provenance) -> None:
    payload = provenance.as_dict()
    print(
        f"{label:24s}: provider={payload['provider']}, dataset={payload['dataset']}, "
        f"reporting_date={payload['reporting_date']}, steps={len(payload['transformation_steps'])}"
    )

summarize_provenance("option diagnostics", diagnostics.provenance)
summarize_provenance("manual rate curve", curve.provenance)
summarize_provenance("portfolio inputs", allocator.context.provenance)
summarize_provenance("backtest", backtest.provenance)
summarize_provenance("credit assessment", credit.provenance)
summarize_provenance("risk dashboard", dashboard.provenance)
summarize_provenance("dashboard report", report.provenance)

In [ ]:
diagnostics.provenance.as_dict()

## 3. Merge provenance from multiple sources

`merge_provenance()` combines several provenance records into one derived
record — useful when one result depends on more than one upstream input
(e.g., a report built from a rate curve and a credit assessment).


In [ ]:
from abaquant.core import merge_provenance

combined = merge_provenance([curve.provenance, credit.provenance])
combined.as_dict()

## Takeaway

For reproducible research, keep four things together: the result object,
its input parameters, its `.provenance` metadata, and the AbaQuant package
version. Provenance explains computational lineage — it does **not**
guarantee provider correctness, licensing compliance, or economic validity.
See `docs/reference/provenance.rst`.
